In [9]:
from datatools import get_price, get_basic
from factortools import MAD_winsorize
import pandas as pd
import numpy as np

In [ ]:
def __reg(df, y, X, W):
    b = np.linalg.pinv(X.T@W@X)@X.T@W@y
# 去除极端值
    resi = MAD_winsorize(y - X@b, multiplier=5)
# 标准化
    resi -= np.nanmean(resi)
    resi /= np.nanstd(resi)
    return pd.Series(resi, index=df['code'])

In [7]:
df = get_basic(start_date='2014-01-01', end_date='2025-12-31',fields=['circ_mv'])

In [ ]:
df['LNCAP'] = np.log(df['circ_mv'])
df['sub_MIDCAP'] = df['LNCAP']**3
# 截面回归正交化处理    
MIDCAP = df.groupby('time').apply(lambda x: __reg(x, x['sub_MIDCAP'].values, np.c_[np.ones((df.shape[1], 1)), x['LNCAP'].values], np.diag(np.sqrt(x['circ_mv']))))
MIDCAP.name = 'MIDCAP'
df = df.merge(MIDCAP.reset_index())

,ts_code,trade_date,circ_mv
0,000001.SZ,2014-01-02,6.819328e+06
1,000002.SZ,2014-01-02,7.730278e+06
2,000004.SZ,2014-01-02,9.905692e+04
3,000005.SZ,2014-01-02,2.275220e+05
4,000006.SZ,2014-01-02,6.506091e+05
...,...,...,...
11276282,920978.BJ,2025-12-31,3.724500e+05
11276283,920981.BJ,2025-12-31,1.192415e+05
11276284,920982.BJ,2025-12-31,1.283132e+06
11276285,920985.BJ,2025-12-31,1.576895e+05
